# Retail Event Count Analysis

This notebook loads the timestamped retail user-behavior events and evaluates the data using the repository's count-window selector logic.

The goal is to determine whether the event stream is suitable for a Poisson-like count model and which aggregation window is most compatible.


In [1]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().resolve()
candidate_roots = [ROOT, *ROOT.parents]
for candidate in candidate_roots:
    if (candidate / "src" / "count_ts_window_selector").exists():
        ROOT = candidate
        break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "retail-user-behavior_timestamp_events.csv"
print(f"Reading {DATA_FILE}")

timestamps = []
with DATA_FILE.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw = row["timestamp"]
        if "." in raw:
            base, frac = raw.split(".", 1)
            raw = base + "." + frac[:6]
        timestamps.append(datetime.strptime(raw, "%Y-%m-%d %H:%M:%S.%f"))

print(f"Rows: {len(timestamps)}")
print(f"Minimum timestamp: {min(timestamps)}")
print(f"Maximum timestamp: {max(timestamps)}")
print(f"Time span (hours): {(max(timestamps) - min(timestamps)).total_seconds() / 3600:.2f}")


Reading /home/rajiv/programming/count_ts_window_selector/data/retail-user-behavior_timestamp_events.csv
Rows: 4032
Minimum timestamp: 2024-02-01 10:00:09.470713
Maximum timestamp: 2024-02-21 17:52:20.169331
Time span (hours): 487.87


In [2]:
hourly_counts = defaultdict(int)
for dt in timestamps:
    key = dt.replace(minute=0, second=0, microsecond=0)
    hourly_counts[key] += 1

series = [hourly_counts[k] for k in sorted(hourly_counts)]
print(f'Hourly bins: {len(series)}')
print(f'Mean hourly count: {sum(series) / len(series):.2f}')
print(f'Min hourly count: {min(series)}')
print(f'Max hourly count: {max(series)}')
print(f'Zero-hour proportion: {sum(v == 0 for v in series) / len(series):.3f}')
print('First 20 hourly counts:', series[:20])


Hourly bins: 168
Mean hourly count: 24.00
Min hourly count: 9
Max hourly count: 38
Zero-hour proportion: 0.000
First 20 hourly counts: [29, 25, 23, 24, 27, 17, 23, 27, 18, 20, 24, 18, 27, 28, 20, 24, 27, 22, 24, 25]


In [3]:
from count_ts_window_selector import (
    WindowSelectionConfig,
    aggregate_counts_by_window,
    check_data_suitability,
    compute_fano_confidence_interval,
    compute_fano_factor,
    select_time_window,
)

config = WindowSelectionConfig(
    min_event_rate=5.0,
    max_zero_window_proportion=0.1,
    min_windows=30,
    confidence_level=0.90,
)

print('Suitability check:', check_data_suitability(series, min_event_rate=config.min_event_rate, max_zero_window_proportion=config.max_zero_window_proportion))

candidate_windows = [1, 2, 3, 4, 6, 8, 12, 24]
results = select_time_window(series, candidate_windows, config=config)
print('Selected Poisson-compatible windows:')
for result in results:
    print(result)


Suitability check: {'observed_event_rate': 24.0, 'zero_window_proportion': 0.0}
Selected Poisson-compatible windows:
{'window_size': 1, 'fano_factor': 0.9980039920159681, 'fano_lower': 0.8270562957765132, 'fano_upper': 1.1865519567146168}
{'window_size': 2, 'fano_factor': 0.9633534136546186, 'fano_lower': 0.7590829929963313, 'fano_upper': 1.268279244540486}
{'window_size': 3, 'fano_factor': 0.7742424242424242, 'fano_lower': 0.7083277550324562, 'fano_upper': 1.3329362368924225}
{'window_size': 4, 'fano_factor': 1.0818089430894309, 'fano_lower': 0.6664768651218096, 'fano_upper': 1.3888387108981484}


## Interpretation

The hourly aggregate is dense enough for the non-hurdle workflow: the mean hourly event count is about 24, and there are no zero-hour windows. The resulting Fano factors stay close to 1 for 1-, 2-, 3-, and 4-hour windows, with a 1-hour window giving a Fano factor of approximately 0.998 and a 90% confidence band of roughly [0.827, 1.187]. That is strong evidence that the process behaves near-Poisson over a 1- to 4-hour observation horizon.

The closest match to the Poisson reference is the 1-hour aggregate, which is therefore the most defensible default observation window for this dataset.
